In [1]:
import pandas as pd
import numpy as np

import joblib
import json

from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 125)


In [3]:
def health_label(score):

    if score < 35:
        return "Low"

    elif score < 60:
        return "Medium"

    else:
        return "High"


df["startup_health_label"] = (
    df["startup_health_score"]
    .apply(health_label)
)

print(
    df["startup_health_label"]
    .value_counts()
)

startup_health_label
Medium    42020
Low        7089
High        891
Name: count, dtype: int64


In [4]:
features = [

    "founder_strength_score",

    "funding_strength_score",

    "market_opportunity_score",

    "growth_score",

    "revenue",

    "customer_count",

    "retention_rate",

    "employee_count",

    "runway_months",

    "burn_rate"

]

X = df[features]

y = df["startup_health_label"]

In [5]:
le = LabelEncoder()

y = le.fit_transform(y)

print(
    dict(
        zip(
            le.classes_,
            range(
                len(le.classes_)
            )
        )
    )
)

{'High': 0, 'Low': 1, 'Medium': 2}


In [6]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

In [7]:
model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    random_state=42,

    n_jobs=-1

)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(max_depth=12, n_estimators=300, n_jobs=-1,
                       random_state=42)

In [8]:
preds = model.predict(X_test)

acc = accuracy_score(
    y_test,
    preds
)

print(
    "Accuracy:",
    round(acc*100,2),
    "%"
)

print(
    classification_report(
        y_test,
        preds
    )
)

Accuracy: 96.34 %
              precision    recall  f1-score   support

           0       1.00      0.29      0.45       178
           1       0.99      0.84      0.91      1418
           2       0.96      1.00      0.98      8404

    accuracy                           0.96     10000
   macro avg       0.98      0.71      0.78     10000
weighted avg       0.96      0.96      0.96     10000



In [9]:
importance = pd.DataFrame({

    "Feature": features,

    "Importance":
        model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

print(
    importance
)

                    Feature  Importance
0    founder_strength_score    0.281923
1    funding_strength_score    0.270330
3              growth_score    0.216178
2  market_opportunity_score    0.113621
6            retention_rate    0.027271
5            customer_count    0.020248
4                   revenue    0.019478
9                 burn_rate    0.019017
7            employee_count    0.016089
8             runway_months    0.015845


In [11]:
joblib.dump(

    model,

    "../models/startup_health_model/startup_health_model.pkl"

)

joblib.dump(

    le,

    "../models/startup_health_model/label_encoder.pkl"

)

print("Model Saved ✅")

Model Saved ✅


In [12]:
metadata = {

    "model_name":
        "Startup Health Model",

    "algorithm":
        "Random Forest",

    "features":
        features,

    "classes":
        list(
            le.classes_
        )

}

with open(

    "../models/startup_health_model/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [13]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("Dataset Saved ✅")

Dataset Saved ✅


In [15]:
print(df["startup_health_label"].value_counts())

print("Accuracy:", acc)

print(importance.head(10))

startup_health_label
Medium    42020
Low        7089
High        891
Name: count, dtype: int64
Accuracy: 0.9634
                    Feature  Importance
0    founder_strength_score    0.281923
1    funding_strength_score    0.270330
3              growth_score    0.216178
2  market_opportunity_score    0.113621
6            retention_rate    0.027271
5            customer_count    0.020248
4                   revenue    0.019478
9                 burn_rate    0.019017
7            employee_count    0.016089
8             runway_months    0.015845
